# Billing NPI Analysis - Quick Reference

## Purpose
Maps MPSII patients to **billing organizations** (hospitals, infusion centers) rather than individual providers.

## What It Creates

### View 1: MPSII_Treatment_Elaprase_Only
- **Contains:** All Elaprase treatment claims (5-year window: Aug 2020 - Jul 2025)
- **Sources:** 
  - Medical claims with NDC codes: '54092070001', '540920700'
  - Medical claims with procedure code: 'J1743'
- **Fields:** PATIENT_ID, BILLING_NPI, BILLING_NPI_CONFIDENCE
- **Excludes:** Pharmacy claims (commented out - no meaningful billing NPI)

### View 2: billing_npi_information
- **Contains:** Billing NPIs enriched with organization details
- **Joins:** View 1 + kom_providers table
- **Fields:** 
  - BILLING_NPI
  - ORGANIZATION_NAME (facility/hospital name)
  - PROVIDER_ADDRESS, PROVIDER_CITY, PROVIDER_STATE, PROVIDER_ZIP
  - PROVIDER_TYPE (organization type)
  - BILLING_NPI_CONFIDENCE (data quality: HIGH/MEDIUM/LOW)
- **Filters:** Excludes NULL billing NPIs

## Key Concept
- **Billing NPI** = Organization that submits the claim (e.g., "Children's Hospital Boston")
- **Rendering NPI** = Individual who delivers care (e.g., "Dr. Smith, Geneticist")

## Use Cases
- Identify high-volume billing organizations
- Geographic distribution of treatment facilities
- Organizational-level targeting for GTM campaigns
- Facility affiliation analysis

## Next Steps
Query these views to get insights:
```sql
-- Patient counts per organization
SELECT BILLING_NPI, ORGANIZATION_NAME, COUNT(DISTINCT PATIENT_ID)
FROM billing_npi_information b
JOIN MPSII_Treatment_Elaprase_Only t ON b.BILLING_NPI = t.BILLING_NPI
GROUP BY BILLING_NPI, ORGANIZATION_NAME
ORDER BY COUNT(DISTINCT PATIENT_ID) DESC;
```

In [0]:
-- Step 1: Create first temporary view
CREATE OR REPLACE TEMPORARY VIEW MPSII_Treatment_Elaprase_Only AS
SELECT DISTINCT PATIENT_ID, BILLING_NPI, BILLING_NPI_CONFIDENCE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001','540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
UNION ALL
SELECT DISTINCT PATIENT_ID, BILLING_NPI, BILLING_NPI_CONFIDENCE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE = 'J1743'
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31';

-- Step 1.5: Get unique reference file HCO Zip per NPI (in case of duplicates, take first non-null)
CREATE OR REPLACE TEMPORARY VIEW reference_hco_zip AS
SELECT 
    `HCO 
Primary NPI` AS HCO_Primary_NPI,
    FIRST(`HCO Zip`, TRUE) AS HCO_Zip
FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_12_12_2025
WHERE `HCO 
Primary NPI` IS NOT NULL
GROUP BY `HCO 
Primary NPI`;

-- Step 2: Create second temporary view with patient count, patient list with confidence prefix, and priority zip logic
CREATE OR REPLACE TEMPORARY VIEW billing_npi_base AS
SELECT 
    c1.BILLING_NPI,
    MAX(p1.ORGANIZATION_NAME) AS ORGANIZATION_NAME,
    MAX(p1.PROVIDER_ADDRESS) AS PROVIDER_ADDRESS,
    MAX(p1.PROVIDER_CITY) AS PROVIDER_CITY,
    MAX(p1.PROVIDER_STATE) AS PROVIDER_STATE,
    MAX(p1.PROVIDER_ZIP) AS PROVIDER_ZIP,
    MAX(p1.PROVIDER_TYPE) AS PROVIDER_TYPE,
    COUNT(DISTINCT c1.PATIENT_ID) AS PATIENTS_BILLED_BY_NPI,
    LISTAGG(CONCAT(c1.PATIENT_ID, ' (', SUBSTRING(c1.BILLING_NPI_CONFIDENCE, 1, 1), ')'), ', ') 
        WITHIN GROUP (ORDER BY c1.PATIENT_ID) AS PATIENT_LIST_WITH_CONFIDENCE,
    COALESCE(MAX(ref.HCO_Zip), MAX(p1.PROVIDER_ZIP)) AS FINAL_ZIP
FROM MPSII_Treatment_Elaprase_Only c1
LEFT JOIN com_edp_prd.com_raw.kom_providers p1
    ON p1.NPI = c1.BILLING_NPI
LEFT JOIN reference_hco_zip ref
    ON ref.HCO_Primary_NPI = c1.BILLING_NPI
WHERE c1.BILLING_NPI IS NOT NULL
GROUP BY c1.BILLING_NPI;

-- Step 3: Add territory and region information with deduplication
CREATE OR REPLACE TEMPORARY VIEW billing_npi_information AS
SELECT DISTINCT
    b.BILLING_NPI,
    b.ORGANIZATION_NAME,
    b.PROVIDER_ADDRESS,
    b.PROVIDER_CITY,
    b.PROVIDER_STATE,
    b.PROVIDER_ZIP,
    b.PROVIDER_TYPE,
    b.PATIENTS_BILLED_BY_NPI,
    b.PATIENT_LIST_WITH_CONFIDENCE,
    zt.territory_name,
    zt.region_name
FROM billing_npi_base b
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping zt
    ON LPAD(CAST(zt.zipcode AS STRING), 5, '0') = LPAD(CAST(b.FINAL_ZIP AS STRING), 5, '0');

In [0]:
select * from com_edp_prd.com_intgr.crx_roster
where employee_first_name ilike '%chris%';

select * from com_edp_prd.com_consm.vw_emp_roster_crx_outbound;

In [0]:
select * from com_edp_prd.com_consm.vw_emp_roster_crx_outbound;

In [0]:
%sql
SELECT * FROM billing_npi_information;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping limit 10

In [0]:
SELECT * FROM com_edp_prd.com_raw.kom_providers
WHERE NPI IN 
(1013455021, 1013374917, 1013564590);
--

--

SELECT * FROM com_edp_prd.com_raw.kom_providers
WHERE NPI IN 
(1598797334, 1417257601, 1952303331);


In [0]:
select zipcode, territory_name from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
where zipcode in (45219,
45601,
45414,
26505,
45005,
44130,
75057,
44512,
44622,
11201)